# QwenPRM Wrapper — Score Inspection

Inspect the `QwenPRM` wrapper class (`reward_models.py`) through
the shared `score(questions, answers, batch_size=...)` interface.

Two examples: (1) the flamingo problem — single question, single
answer; (2) a batched call combining the flamingo question with
the algebra correct/wrong pair — two questions, mixed answer
counts, scored in one `prm.score()` call.

Note: the model card specifies `bfloat16`, but the V100 (sm_70)
has no bf16 support, so we load `float16`. fp16 preserves step
*rankings* but can drift absolute scores slightly.

Env: runs under `py311` (transformers 4.57). The bundled remote
code (`modeling_qwen2_rm.py`) calls a cache API removed in newer
transformers; `QwenPRM` uses `use_cache=False` to sidestep it.

## Setup

In [ ]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL+1)
logging.disable(logging.CRITICAL)

import gc

import torch

from notebook_utils import gpu_mem_used_gb, print_step_scores
from reward_models import QwenPRM

In [ ]:
# Model paths
base_dir = "/groups/chichengz/tnn/datasets"
prm_dir = f"{base_dir}/Qwen2.5-Math-PRM-7B"

## Load the PRM

In [ ]:
# fp16 for V100 (sm_70); model card recommends bf16 (Ampere+),
# so absolute scores may drift slightly on Ampere GPUs.
prm = QwenPRM(prm_dir)

print(f"sep token id  : {prm.sep_token_id}")
print(f"dtype         : {next(prm.model.parameters()).dtype}")
print(f"GPU memory used: {gpu_mem_used_gb():.2f} GB")

## Example 1 — flamingo problem (single question, single answer)

Model-card reference scores (bf16): `[1.0, 0.1904, 0.9766, 1.0]`.
On V100/fp16 expect close, not exact.

In [ ]:
# Toy example from the Qwen2.5-Math-PRM-7B model card.
# Double-backslash LaTeX so "\t"/"\b" don't become control chars.
problem = (
    "Sue lives in a fun neighborhood.  One weekend, the "
    "neighbors decided to play a prank on Sue.  On Friday "
    "morning, the neighbors placed 18 pink plastic flamingos "
    "out on Sue's front yard.  On Saturday morning, the "
    "neighbors took back one third of the flamingos, painted "
    "them white, and put these newly painted white flamingos "
    "back out on Sue's front yard.  Then, on Sunday morning, "
    "they added another 18 pink plastic flamingos to the "
    "collection. At noon on Sunday, how many more pink "
    "plastic flamingos were out than white plastic flamingos?"
)

reasoning_steps = [
    "To find out how many more pink plastic flamingos were "
    "out than white plastic flamingos at noon on Sunday, we "
    "can break down the problem into steps. First, on Friday, "
    "the neighbors start with 18 pink plastic flamingos.",

    "On Saturday, they take back one third of the flamingos. "
    "Since there were 18 flamingos, (1/3 \\times 18 = 6) "
    "flamingos are taken back. So, they have (18 - 6 = 12) "
    "flamingos left in their possession. Then, they paint "
    "these 6 flamingos white and put them back out on Sue's "
    "front yard. Now, Sue has the original 12 pink flamingos "
    "plus the 6 new white ones. Thus, by the end of Saturday, "
    "Sue has (12 + 6 = 18) pink flamingos and 6 white "
    "flamingos.",

    "On Sunday, the neighbors add another 18 pink plastic "
    "flamingos to Sue's front yard. By the end of Sunday "
    "morning, Sue has (18 + 18 = 36) pink flamingos and "
    "still 6 white flamingos.",

    "To find the difference, subtract the number of white "
    "flamingos from the number of pink flamingos: "
    "(36 - 6 = 30). Therefore, at noon on Sunday, there were "
    "30 more pink plastic flamingos out than white plastic "
    "flamingos. The answer is (\\boxed{30}).",
]

In [ ]:
# One question, one candidate answer whose steps are joined by
# "\n\n". score() returns [question][answer][step].
scores = prm.score([problem], [["\n\n".join(reasoning_steps)]])

print("=== Flamingo trajectory ===")
print_step_scores(reasoning_steps, scores[0][0])

## Example 2 — batched call (two questions, mixed answer counts)

One `prm.score()` call with two questions: the flamingo problem
(one answer) and the algebra problem (two candidate answers —
correct and wrong). The base class flattens all pairs, scores in
one batched forward pass, and reshapes back to
`[question][answer][step]`.

In [ ]:
algebra_problem = "If 3x + 5 = 17, what is x?"

correct_steps = [
    "We need solve the equation 3x + 5 = 17.",
    "Subtracting 5 from both sides gives 3x = 12.",
    "Dividing both sides by 3 gives x = 4.",
    "Therefore, the answer is (\\boxed{4}).",
]

wrong_steps = [
    "Subtracting 5 from both sides gives 3x = 12.",
    "Dividing both sides by 2 gives x = 6.",
    "Therefore, the final answer is \\boxed{6}.",
]

In [ ]:
# Two questions; flamingo has 1 answer, algebra has 2.
# score() flattens to 3 pairs, batches, reshapes back.
questions = [problem, algebra_problem]
answers = [
    ["\n\n".join(reasoning_steps)],
    ["\n\n".join(correct_steps), "\n\n".join(wrong_steps)],
]

batch_scores = prm.score(questions, answers, batch_size=4)

print("=== Flamingo (Q0, A0) ===")
print_step_scores(reasoning_steps, batch_scores[0][0])

print("\n=== Algebra correct (Q1, A0) ===")
print_step_scores(correct_steps, batch_scores[1][0])

print("\n=== Algebra wrong — step 2 divides by 2 (Q1, A1) ===")
print_step_scores(wrong_steps, batch_scores[1][1])

## Cleanup

In [ ]:
del prm
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory used: {gpu_mem_used_gb():.2f} GB")